# Parameter Sensitivity Runner — patched

**Two-pass design** because RFdiffusion/ProteinMPNN and ESMFold can have incompatible dependencies.

- **Pass 1:** run Cells 0 → 1 → 2 → 3 → 4
- **Restart runtime**
- **Pass 2:** run Cells 0 → 1 → 5 → 6
- **Summary:** run Cell 7 after both passes

Patched to add GPU/dependency preflight checks, use geometry-filtered RFdiffusion outputs for ProteinMPNN, fail early if binder chain is not B, store ESMFold model on Drive, guard against unexpectedly long sequences, and make ESMFold resume-safe.

In [ ]:
# ============================================================
# Cell 0: Mount Drive, set paths, load conditions
# RUN IN BOTH PASSES
# ============================================================
from google.colab import drive

drive.mount('/content/drive')

import os, glob, re, subprocess, time, shutil, sys, gc
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/drive/Othercomputers/My Mac/pdl1-mini-binder-design"
TARGET_PDB = f"{PROJECT_DIR}/data/structures/pdl1_clean.pdb"
STUDY_DIR = f"{PROJECT_DIR}/data/results/param_sensitivity"
CONDITIONS_CSV = "/content/drive/MyDrive/Colab Notebooks/param_study/conditions.csv"
MODEL_DIR = f"{STUDY_DIR}/models"

os.makedirs(STUDY_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

assert os.path.exists(PROJECT_DIR), f"Missing project directory: {PROJECT_DIR}"
assert os.path.exists(TARGET_PDB), f"Missing target PDB: {TARGET_PDB}"
assert os.path.exists(CONDITIONS_CSV), f"Missing conditions CSV: {CONDITIONS_CSV}"

conditions = pd.read_csv(CONDITIONS_CSV)
required_cols = {'condition_id','binder_length','hotspot_residues','target_range','num_designs','noise_scale_ca','noise_scale_frame'}
missing = required_cols - set(conditions.columns)
assert not missing, f"conditions.csv missing columns: {sorted(missing)}"

print(f"Loaded {len(conditions)} conditions")
print(conditions[['condition_id', 'binder_length', 'noise_scale_ca']].to_string(index=False))
print("✓ Ready")

Mounted at /content/drive
Loaded 24 conditions
              condition_id  binder_length  noise_scale_ca
     len25_clusterA_noise0             25             0.0
    len25_clusterA_noise05             25             0.5
     len25_clusterB_noise0             25             0.0
    len25_clusterB_noise05             25             0.5
  len25_distributed_noise0             25             0.0
 len25_distributed_noise05             25             0.5
     len50_clusterA_noise0             50             0.0
    len50_clusterA_noise05             50             0.5
     len50_clusterB_noise0             50             0.0
    len50_clusterB_noise05             50             0.5
  len50_distributed_noise0             50             0.0
 len50_distributed_noise05             50             0.5
     len70_clusterA_noise0             70             0.0
    len70_clusterA_noise05             70             0.5
     len70_clusterB_noise0             70             0.0
    len70_clusterB_noise0

In [ ]:
# ============================================================
# Cell 1: Common imports + GPU preflight
# RUN IN BOTH PASSES
# ============================================================
!pip install -q biopython

import gc, os, sys, subprocess
import torch
from Bio.PDB import PDBParser, NeighborSearch, Superimposer

parser = PDBParser(QUIET=True)

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Colab, select Runtime → Change runtime type → T4 GPU before running paid/limited compute.")

gpu_name = torch.cuda.get_device_name(0)
free, total = torch.cuda.mem_get_info()
print("GPU:", gpu_name)
print(f"GPU memory free/total: {free/1e9:.2f} / {total/1e9:.2f} GB")

if "T4" not in gpu_name:
    print("WARNING: This runtime is not a T4. It may still work, but this notebook was reviewed for T4-class Colab GPUs.")

for p in [PROJECT_DIR, TARGET_PDB, CONDITIONS_CSV, STUDY_DIR]:
    print(f"exists: {p} -> {os.path.exists(p)}")

print("\nConditions preview:")
print(conditions.head().to_string(index=False))
print("\nCondition dtypes:")
print(conditions.dtypes)
print("✓ Imports and preflight complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.1 MB/s eta 0:00:00
Python: 3.12.13
Torch: 2.11.0+cu128
Torch CUDA: 12.8
CUDA available: True
GPU: Tesla T4
GPU memory free/total: 15.53 / 15.64 GB
exists: /content/drive/Othercomputers/My Mac/pdl1-mini-binder-design -> True
exists: /content/drive/Othercomputers/My Mac/pdl1-mini-binder-design/data/structures/pdl1_clean.pdb -> True
exists: /content/drive/MyDrive/Colab Notebooks/param_study/conditions.csv -> True
exists: /content/drive/Othercomputers/My Mac/pdl1-mini-binder-design/data/results/param_sensitivity -> True

Conditions preview:
            condition_id  binder_length                    hotspot_residues  noise_scale_ca  noise_scale_frame  num_designs target_range
   len25_clusterA_noise0             25                   A18,A20,A120,A122             0.0                0.0           20      A18-132
  len25_clusterA_noise05             25                   A18,A20,A120,A122             0.5                0.5           20  

In [ ]:
# ============================================================
# Cell 2: Install RFdiffusion + ProteinMPNN
# PASS 1 ONLY — skip in Pass 2
# ============================================================
import torch, os, subprocess, sys

print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

# --- RFdiffusion ---
if not os.path.exists('/content/RFdiffusion'):
    !git clone https://github.com/RosettaCommons/RFdiffusion.git /content/RFdiffusion

%cd /content/RFdiffusion
!pip install -q hydra-core omegaconf pyrsistent biopython e3nn

# DGL is the most fragile install. This wheel target matches the original working notebook.
# If this import check fails, stop here rather than burning RFdiffusion compute.
!pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html

try:
    import dgl
    print("DGL:", dgl.__version__)
except Exception as e:
    raise RuntimeError(
        "DGL installed but failed to import. This usually means a Torch/CUDA/DGL wheel mismatch. "
        "Check torch.__version__ and torch.version.cuda above before running RFdiffusion."
    ) from e

!pip install -q -e .

if not os.path.exists('/content/RFdiffusion/DeepLearningExamples'):
    !git clone https://github.com/NVIDIA/DeepLearningExamples.git /content/RFdiffusion/DeepLearningExamples

%cd /content/RFdiffusion/DeepLearningExamples/DGLPyTorch/DrugDiscovery/SE3Transformer
!pip install -q -e .

%cd /content/RFdiffusion
!mkdir -p models
!wget -q -nc -P models/ http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt
!wget -q -nc -P models/ http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt
print("✓ RFdiffusion setup complete")

# --- ProteinMPNN ---
if not os.path.exists('/content/ProteinMPNN'):
    !git clone https://github.com/dauparas/ProteinMPNN.git /content/ProteinMPNN

%cd /content/ProteinMPNN
!ls protein_mpnn_run.py

!python -c "import torch; print('ProteinMPNN CUDA check:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"

print("✓ ProteinMPNN ready")

Torch: 2.11.0+cu128
Torch CUDA: 12.8
GPU: Tesla T4
Cloning into '/content/RFdiffusion'...
remote: Enumerating objects: 753, done.
remote: Counting objects: 100% (546/546), done.
remote: Compressing objects: 100% (340/340), done.
remote: Total 753 (delta 337), reused 238 (delta 202), pack-reused 207 (from 1)
Receiving objects: 100% (753/753), 10.19 MiB | 17.22 MiB/s, done.
Resolving deltas: 100% (385/385), done.
/content/RFdiffusion
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.3/122.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.7/450.7 kB 32.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.5/370.5 MB 2.9 MB/s eta 0:00:00


DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)
DGL: 2.5.0+cu121
  Preparing metadata (setup.py) ... done
ERROR: Could not find a version that satisfies the requirement se3-transformer (from rfdiffusion) (from versions: none)
ERROR: No matching distribution found for se3-transformer
Cloning into '/content/RFdiffusion/DeepLearningExamples'...
remote: Enumerating objects: 33828, done.
remote: Total 33828 (delta 0), reused 0 (delta 0), pack-reused 33828 (from 1)
Receiving objects: 100% (33828/33828), 110.22 MiB | 14.43 MiB/s, done.
Resolving deltas: 100% (23838/23838), done.
/content/RFdiffusion/DeepLearningExamples/DGLPyTorch/DrugDiscovery/SE3Transformer
  Preparing metadata (setup.py) ... done
/content/RFdiffusion
✓ RFdiffusion setup complete
Cloning into '/content/ProteinMPNN'...
remote: Enumerating objects: 634, done.
remote: Counting

In [ ]:
# ============================================================
# Cell 3: Helper functions for Pass 1 (RFdiffusion + ProteinMPNN)
# PASS 1 ONLY
# ============================================================

def run_rfdiffusion_condition(row):
    condition_id = row['condition_id']
    binder_length = int(row['binder_length'])
    hotspot_residues = str(row['hotspot_residues'])
    target_range = str(row['target_range'])
    num_designs = int(row['num_designs'])
    noise_scale_ca = row['noise_scale_ca']
    noise_scale_frame = row['noise_scale_frame']
    output_dir = f"{STUDY_DIR}/{condition_id}/rfdiffusion_outputs"
    os.makedirs(output_dir, exist_ok=True)
    os.chdir('/content/RFdiffusion')
    sys.path.append('/content/RFdiffusion')
    os.environ['PYTHONPATH'] = '/content/RFdiffusion:' + os.environ.get('PYTHONPATH', '')
    existing = set(f for f in os.listdir(output_dir) if f.endswith('.pdb') and 'traj' not in f)
    print(f"  RFdiffusion: {len(existing)}/{num_designs} already done")
    if len(existing) >= num_designs:
        return True
    for i in range(num_designs):
        expected = f"design_{i}_0.pdb"
        if expected in existing:
            continue
        cmd = [
            'python', 'scripts/run_inference.py',
            f'inference.output_prefix={output_dir}/design_{i}',
            f'inference.input_pdb={TARGET_PDB}',
            f'contigmap.contigs=[{binder_length}-{binder_length}/0 {target_range}]',
            f'ppi.hotspot_res=[{hotspot_residues}]',
            'inference.num_designs=1',
            f'denoiser.noise_scale_ca={noise_scale_ca}',
            f'denoiser.noise_scale_frame={noise_scale_frame}',
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, env=os.environ.copy())
        if result.returncode != 0:
            print(f"    Design {i} failed: {result.stderr[-1000:]}")
            return False
        if (i + 1) % 5 == 0 or i == 0:
            print(f"    {i+1}/{num_designs}")
    print("  ✓ RFdiffusion complete")
    return True


def filter_backbone_geometry(condition_id):
    output_dir = f"{STUDY_DIR}/{condition_id}/rfdiffusion_outputs"
    design_files = sorted([f for f in glob.glob(f"{output_dir}/design_*.pdb") if 'traj' not in f])
    print(f"  Geometry filter: {len(design_files)} PDB files")
    quality_data = []
    for pdb_file in design_files:
        try:
            structure = parser.get_structure('design', pdb_file)
            mdl = structure[0]
            chains = list(mdl.get_chains())
            chain_sizes = [(c, len([r for r in c.get_residues() if r.id[0] == ' '])) for c in chains]
            chain_sizes.sort(key=lambda x: x[1])
            binder_chain = chain_sizes[0][0]
            target_chain = chain_sizes[-1][0]
            binder_residues = [r for r in binder_chain.get_residues() if r.id[0] == ' ']
            ca_coords = np.array([r['CA'].get_vector().get_array() for r in binder_residues if 'CA' in r])
            if len(ca_coords) == 0:
                continue
            centroid = ca_coords.mean(axis=0)
            rg = np.sqrt(np.mean(np.sum((ca_coords - centroid)**2, axis=1)))
            target_atoms = list(target_chain.get_atoms())
            ns = NeighborSearch(target_atoms)
            contact_binder = set(); contact_target = set()
            for residue in binder_residues:
                for atom in residue.get_atoms():
                    neighbors = ns.search(atom.get_vector().get_array(), 4.5)
                    if neighbors:
                        contact_binder.add(residue.id[1])
                        for n in neighbors:
                            p = n.get_parent()
                            if p.id[0] == ' ':
                                contact_target.add(p.id[1])
            quality_data.append({
                'file': os.path.basename(pdb_file), 'pdb_path': pdb_file,
                'backbone': os.path.basename(pdb_file).replace('.pdb', ''),
                'binder_chain': binder_chain.id, 'target_chain': target_chain.id,
                'n_residues': len(binder_residues), 'radius_of_gyration': round(rg, 2),
                'binder_contact_residues': len(contact_binder),
                'target_contact_residues': len(contact_target),
            })
        except Exception as e:
            print(f"    Error: {pdb_file}: {e}")
    df = pd.DataFrame(quality_data)
    if len(df) == 0:
        print("  No designs to filter")
        return False
    all_metrics_csv = f"{output_dir}/all_design_geometry_metrics.csv"
    filtered_csv = f"{output_dir}/filtered_designs.csv"
    filtered_pdb_dir = f"{output_dir}/filtered_pdbs"
    df.to_csv(all_metrics_csv, index=False)
    df_good = df[(df['radius_of_gyration'] > 8) & (df['radius_of_gyration'] < 20) & (df['binder_contact_residues'] >= 3) & (df['target_contact_residues'] >= 2)].copy()
    df_good.to_csv(filtered_csv, index=False)
    if os.path.exists(filtered_pdb_dir):
        shutil.rmtree(filtered_pdb_dir)
    os.makedirs(filtered_pdb_dir, exist_ok=True)
    for _, r in df_good.iterrows():
        shutil.copy2(r['pdb_path'], f"{filtered_pdb_dir}/{r['file']}")
    print(f"  ✓ Geometry: {len(df_good)}/{len(df)} pass")
    print(f"  Filtered PDBs for ProteinMPNN: {filtered_pdb_dir}")
    return len(df_good) > 0


def run_proteinmpnn_condition(condition_id, seqs_per_design=8, temperature=0.1):
    rfd_dir = f"{STUDY_DIR}/{condition_id}/rfdiffusion_outputs"
    input_dir = f"{rfd_dir}/filtered_pdbs"
    output_dir = f"{STUDY_DIR}/{condition_id}/proteinmpnn"
    seq_csv = f"{output_dir}/all_mpnn_sequences.csv"
    filtered_csv = f"{rfd_dir}/filtered_designs.csv"
    if os.path.exists(seq_csv):
        print("  ProteinMPNN: already complete, skipping")
        return True
    if not os.path.exists(filtered_csv):
        print("  Missing filtered_designs.csv. Run geometry filter first.")
        return False
    df_filtered = pd.read_csv(filtered_csv)
    if len(df_filtered) == 0:
        print("  No geometry-passing backbones for ProteinMPNN")
        return False
    binder_chains = sorted(df_filtered['binder_chain'].dropna().unique().tolist())
    if binder_chains != ['B']:
        print(f"  STOP: expected binder chain B for ProteinMPNN, but geometry found binder chains: {binder_chains}")
        print("  Inspect RFdiffusion output or modify ProteinMPNN chain assignment before proceeding.")
        return False
    os.makedirs(output_dir, exist_ok=True)
    os.chdir('/content/ProteinMPNN')
    parsed_jsonl = f"{output_dir}/parsed_pdbs.jsonl"
    assigned_jsonl = f"{output_dir}/assigned_pdbs.jsonl"
    subprocess.run(['python', 'helper_scripts/parse_multiple_chains.py', '--input_path', input_dir, '--output_path', parsed_jsonl], check=True)
    subprocess.run(['python', 'helper_scripts/assign_fixed_chains.py', '--input_path', parsed_jsonl, '--output_path', assigned_jsonl, '--chain_list', 'B'], check=True)
    result = subprocess.run(['python', 'protein_mpnn_run.py', '--jsonl_path', parsed_jsonl, '--chain_id_jsonl', assigned_jsonl, '--out_folder', output_dir, '--num_seq_per_target', str(seqs_per_design), '--sampling_temp', str(temperature), '--batch_size', '1'], capture_output=True, text=True, env=os.environ.copy())
    if result.returncode != 0:
        print(f"  ProteinMPNN failed: {result.stderr[-1000:]}")
        return False
    fasta_files = sorted(glob.glob(f"{output_dir}/seqs/*.fa"))
    all_designs = []
    for fasta in fasta_files:
        backbone = os.path.basename(fasta).replace('.fa', '')
        with open(fasta) as f:
            lines = [x.strip() for x in f.readlines() if x.strip()]
        for i, line in enumerate(lines):
            if line.startswith('>') and i + 1 < len(lines):
                header = line; sequence = lines[i + 1]
                binder_seq = sequence.split('/')[-1] if '/' in sequence else sequence
                score_match = re.search(r'score=([-0-9.]+)', header)
                global_match = re.search(r'global_score=([-0-9.]+)', header)
                all_designs.append({'condition_id': condition_id, 'backbone': backbone, 'header': header, 'full_sequence': sequence, 'binder_sequence': binder_seq, 'binder_length': len(binder_seq), 'mpnn_score': float(score_match.group(1)) if score_match else None, 'global_score': float(global_match.group(1)) if global_match else None})
    df = pd.DataFrame(all_designs)
    if len(df) == 0:
        print("  ProteinMPNN produced no parsed sequences")
        return False
    df.to_csv(seq_csv, index=False)
    print(f"  ✓ ProteinMPNN: {len(df)} sequences from {df['backbone'].nunique()} geometry-passing backbones")
    return True

print("✓ Pass 1 helpers loaded")

✓ Pass 1 helpers loaded


In [ ]:
# ============================================================
# Cell 4: PASS 1 — Run RFdiffusion + ProteinMPNN for all conditions
# Resume-safe. Restart and re-run anytime.
# PASS 1 ONLY
# ============================================================
print(f"Pass 1: RFdiffusion + ProteinMPNN for {len(conditions)} conditions\n")
for idx, row in conditions.iterrows():
    cond_id = row['condition_id']
    print(f"=== [{idx+1}/{len(conditions)}] {cond_id} ===")
    ok = run_rfdiffusion_condition(row)
    if not ok:
        print(f"  FAILED at RFdiffusion, skipping {cond_id}")
        continue
    ok = filter_backbone_geometry(cond_id)
    if not ok:
        print(f"  FAILED or no passing designs at geometry filter, skipping {cond_id}")
        continue
    ok = run_proteinmpnn_condition(cond_id)
    if not ok:
        print(f"  FAILED at ProteinMPNN, skipping {cond_id}")
        continue
    print(f"  ✓ {cond_id} Pass 1 complete\n")
print("\n=== Pass 1 complete. RESTART RUNTIME, then run Cells 0, 1, 5, 6 ===")

Pass 1: RFdiffusion + ProteinMPNN for 24 conditions

=== [1/24] len25_clusterA_noise0 ===
  RFdiffusion: 20/20 already done
  Geometry filter: 20 PDB files
  ✓ Geometry: 2/20 pass
  Filtered PDBs for ProteinMPNN: /content/drive/Othercomputers/My Mac/pdl1-mini-binder-design/data/results/param_sensitivity/len25_clusterA_noise0/rfdiffusion_outputs/filtered_pdbs
  ProteinMPNN: already complete, skipping
  ✓ len25_clusterA_noise0 Pass 1 complete

=== [2/24] len25_clusterA_noise05 ===
  RFdiffusion: 20/20 already done
  Geometry filter: 20 PDB files
  ✓ Geometry: 0/20 pass
  Filtered PDBs for ProteinMPNN: /content/drive/Othercomputers/My Mac/pdl1-mini-binder-design/data/results/param_sensitivity/len25_clusterA_noise05/rfdiffusion_outputs/filtered_pdbs
  FAILED or no passing designs at geometry filter, skipping len25_clusterA_noise05
=== [3/24] len25_clusterB_noise0 ===
  RFdiffusion: 20/20 already done
  Geometry filter: 20 PDB files
  ✓ Geometry: 0/20 pass
  Filtered PDBs for ProteinMPNN: /

In [ ]:
# ============================================================
# Cell 5: Install ESMFold
# PASS 2 ONLY — run after runtime restart
# ============================================================
import os, time, subprocess
MODEL_NAME = "esmfold.model"
MODEL_PATH = f"{MODEL_DIR}/{MODEL_NAME}"
if not os.path.isfile(MODEL_PATH):
    print("Installing aria2...")
    subprocess.run("apt-get install aria2 -qq", shell=True, check=False)
    print("Downloading ESMFold model weights to Drive...")
    subprocess.run(f"aria2c -q -x 16 -d '{MODEL_DIR}' -o '{MODEL_NAME}' https://colabfold.steineggerlab.workers.dev/esm/{MODEL_NAME}", shell=True, check=True)
if not os.path.isfile("finished_install_esmfold"):
    print("Installing Python dependencies...")
    subprocess.run("pip install -q omegaconf pytorch_lightning biopython ml_collections einops py3Dmol modelcif", shell=True, check=True)
    print("Installing NVIDIA dllogger...")
    subprocess.run("pip install -q git+https://github.com/NVIDIA/dllogger.git", shell=True, check=True)
    print("Installing OpenFold...")
    subprocess.run("pip install -q git+https://github.com/sokrypton/openfold.git", shell=True, check=True)
    print("Installing ESMFold...")
    subprocess.run("pip install -q git+https://github.com/sokrypton/esm.git", shell=True, check=True)
    subprocess.run("touch finished_install_esmfold", shell=True, check=True)
assert os.path.isfile(MODEL_PATH), f"Missing ESMFold model: {MODEL_PATH}"
model_size_gb = os.path.getsize(MODEL_PATH) / 1e9
print("✓ ESMFold dependencies installed")
print(f"✓ ESMFold model weights available: {MODEL_PATH} ({model_size_gb:.2f} GB)")

In [ ]:
# ============================================================
# Cell 6: PASS 2 — Run ESMFold + RMSD for all conditions
# PASS 2 ONLY — after runtime restart + Cells 0, 1, 5
# ============================================================
import gc, torch, os
from Bio.PDB import PDBParser, Superimposer
parser = PDBParser(QUIET=True)

def mean_ca_plddt_from_pdb(pdb_path):
    values = []
    with open(pdb_path) as f:
        for line in f:
            if line.startswith('ATOM') and line[12:16].strip() == 'CA':
                try:
                    values.append(float(line[60:66]))
                except ValueError:
                    pass
    return round(sum(values) / len(values), 2) if values else 0

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device != "cuda":
    raise RuntimeError("ESMFold requires GPU. Stop here.")
free, total = torch.cuda.mem_get_info()
print(f"GPU memory free/total before ESMFold load: {free/1e9:.2f} / {total/1e9:.2f} GB")
gc.collect(); torch.cuda.empty_cache()
model = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model = model.eval().cuda()
torch.cuda.empty_cache(); gc.collect()
print("✓ ESMFold model loaded")

print(f"\nPass 2: ESMFold + RMSD for {len(conditions)} conditions\n")
for idx, row in conditions.iterrows():
    cond_id = row['condition_id']
    print(f"=== [{idx+1}/{len(conditions)}] {cond_id} ===")
    seq_csv = f"{STUDY_DIR}/{cond_id}/proteinmpnn/all_mpnn_sequences.csv"
    esm_dir = f"{STUDY_DIR}/{cond_id}/esmfold_predictions"
    pred_csv = f"{esm_dir}/esmfold_binder_predictions.csv"
    val_csv = f"{esm_dir}/validation_results.csv"
    if not os.path.exists(seq_csv):
        print("  No ProteinMPNN output, skipping")
        continue
    if os.path.exists(val_csv):
        print("  Already complete, skipping")
        continue
    os.makedirs(esm_dir, exist_ok=True)
    df_seq = pd.read_csv(seq_csv)
    if len(df_seq) == 0:
        print("  Empty ProteinMPNN sequence CSV, skipping")
        continue
    max_len = int(df_seq['binder_length'].max())
    print(f"  Binder length range: {df_seq['binder_length'].min()}–{max_len}")
    assert max_len < 300, f"Unexpectedly long binder sequence sent to ESMFold: max length {max_len}"
    df_real = df_seq[df_seq['mpnn_score'] < 1.5].copy()
    df_candidates = df_real.sort_values('mpnn_score').drop_duplicates('backbone').reset_index(drop=True)
    print(f"  ESMFold: {len(df_candidates)} candidates")
    if len(df_candidates) == 0:
        print("  No candidates after mpnn_score filter, skipping")
        continue
    prediction_rows = []
    for i, (_, r) in enumerate(df_candidates.iterrows()):
        backbone = r['backbone']; seq = r['binder_sequence']
        seq_id = f"{backbone}_rank{i}"
        output_pdb = f"{esm_dir}/{seq_id}.pdb"
        try:
            if os.path.exists(output_pdb):
                mean_plddt = mean_ca_plddt_from_pdb(output_pdb)
                print(f"    Existing: {seq_id}: pLDDT={mean_plddt:.1f}")
            else:
                with torch.no_grad():
                    pdb_string = model.infer_pdb(seq)
                with open(output_pdb, 'w') as f:
                    f.write(pdb_string)
                mean_plddt = mean_ca_plddt_from_pdb(output_pdb)
                print(f"    {i+1}/{len(df_candidates)} {seq_id}: pLDDT={mean_plddt:.1f}")
                del pdb_string; torch.cuda.empty_cache(); gc.collect()
            prediction_rows.append({'condition_id': cond_id, 'seq_id': seq_id, 'backbone': backbone, 'binder_sequence': seq, 'binder_length': len(seq), 'mpnn_score': r['mpnn_score'], 'global_score': r['global_score'], 'mean_plddt': mean_plddt, 'predicted_pdb': output_pdb})
        except Exception as e:
            print(f"    Failed {seq_id}: {e}")
    df_pred = pd.DataFrame(prediction_rows)
    if len(df_pred) == 0:
        print("  No ESMFold predictions available, skipping RMSD")
        continue
    df_pred.to_csv(pred_csv, index=False)
    print(f"  ✓ ESMFold table: {len(df_pred)} predictions")
    backbone_dir = f"{STUDY_DIR}/{cond_id}/rfdiffusion_outputs"
    rmsd_results = []
    for _, pr in df_pred.iterrows():
        seq_id = pr['seq_id']; backbone = pr['backbone']
        predicted_pdb = pr['predicted_pdb']; designed_pdb = f"{backbone_dir}/{backbone}.pdb"
        if not os.path.exists(predicted_pdb) or not os.path.exists(designed_pdb):
            continue
        try:
            pred_struct = parser.get_structure('pred', predicted_pdb)
            design_struct = parser.get_structure('design', designed_pdb)
            pred_chain = list(pred_struct[0].get_chains())[0]
            if 'B' in [c.id for c in design_struct[0].get_chains()]:
                design_chain = design_struct[0]['B']
            else:
                chains = sorted(list(design_struct[0].get_chains()), key=lambda c: len([r for r in c.get_residues() if r.id[0] == ' ']))
                design_chain = chains[0]
            pred_ca = [a for r in pred_chain.get_residues() for a in r.get_atoms() if a.name == 'CA']
            design_ca = [a for r in design_chain.get_residues() for a in r.get_atoms() if a.name == 'CA']
            n_atoms = min(len(pred_ca), len(design_ca))
            if n_atoms < 10:
                continue
            sup = Superimposer(); sup.set_atoms(design_ca[:n_atoms], pred_ca[:n_atoms])
            rmsd_results.append({'condition_id': cond_id, 'seq_id': seq_id, 'backbone': backbone, 'mean_plddt': pr['mean_plddt'], 'mpnn_score': pr['mpnn_score'], 'global_score': pr['global_score'], 'rmsd': round(sup.rms, 2), 'n_aligned': n_atoms, 'predicted_pdb': predicted_pdb, 'designed_pdb': designed_pdb})
        except Exception as e:
            print(f"    RMSD failed {seq_id}: {e}")
    df_rmsd = pd.DataFrame(rmsd_results)
    df_rmsd.to_csv(val_csv, index=False)
    n_pass = ((df_rmsd['mean_plddt'] >= 80) & (df_rmsd['rmsd'] <= 2.0)).sum() if len(df_rmsd) > 0 else 0
    print(f"  ✓ RMSD: {n_pass}/{len(df_rmsd)} pass\n")

del model; gc.collect(); torch.cuda.empty_cache()
print("=== Pass 2 complete. Run Cell 7 for summary. ===")

In [ ]:
# ============================================================
# Cell 7: Merge results + summary
# Run after BOTH passes. CPU only.
# ============================================================
all_rows = []
for cond_id in conditions['condition_id']:
    val_csv = f"{STUDY_DIR}/{cond_id}/esmfold_predictions/validation_results.csv"
    geo_csv = f"{STUDY_DIR}/{cond_id}/rfdiffusion_outputs/all_design_geometry_metrics.csv"
    if not os.path.exists(val_csv):
        print(f"Missing: {cond_id}")
        continue
    df_val = pd.read_csv(val_csv)
    if os.path.exists(geo_csv):
        df_geo = pd.read_csv(geo_csv)
        df_val = df_val.merge(df_geo, on='backbone', how='left', suffixes=('', '_rfd'))
    all_rows.append(df_val)
if not all_rows:
    print("No validation results yet")
else:
    df_all = pd.concat(all_rows, ignore_index=True)
    df_all.to_csv(f"{STUDY_DIR}/all_conditions_validation_results.csv", index=False)
    validated = df_all[(df_all['mean_plddt'] >= 80) & (df_all['rmsd'] <= 2.0)]
    ranked = validated.sort_values(['rmsd', 'mean_plddt'], ascending=[True, False]).reset_index(drop=True)
    ranked.index += 1; ranked.index.name = 'rank'
    ranked.to_csv(f"{STUDY_DIR}/final_ranked_candidates.csv")
    summary = df_all.groupby('condition_id').agg(n_tested=('seq_id', 'count'), mean_plddt=('mean_plddt', 'mean'), mean_rmsd=('rmsd', 'mean'), rmsd_min=('rmsd', 'min')).reset_index()
    val_counts = validated.groupby('condition_id').size().reset_index(name='n_pass')
    summary = summary.merge(val_counts, on='condition_id', how='left')
    summary['n_pass'] = summary['n_pass'].fillna(0).astype(int)
    summary.to_csv(f"{STUDY_DIR}/condition_summary.csv", index=False)
    print(f"Total: {len(df_all)} designs, {len(validated)} pass")
    print(summary.sort_values(['n_pass', 'rmsd_min'], ascending=[False, True]).to_string(index=False))